In [69]:
import pandas as pd
import numpy as np
import catboost
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
import xgboost
print("Using CatBoost version",catboost.__version__)
print("Using XGBoost version",xgboost.__version__)
import sklearn
print(sklearn.__version__)
from sklearn.metrics import mean_squared_log_error, make_scorer
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestRegressor,GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import LabelEncoder,OrdinalEncoder
from sklearn.svm import SVR
from sklearn.linear_model import SGDRegressor
from sklearn.decomposition import PCA
from sklearn.model_selection import cross_val_score,KFold
from sklearn import set_config
set_config(display='text')
import warnings
warnings.filterwarnings("ignore")

Using CatBoost version 1.2.7
Using XGBoost version 2.1.3
1.6.1


In [3]:
train_df = pd.read_csv("train_molecular.csv")
train_df.head()

,Batch_ID,T80,Smiles,Mass,HAcceptors,HDonors,LogP,Asphericity,Rg,TPSA,...,SDOS4.5,SDOS4.6,SDOS4.7,SDOS4.8,SDOS4.9,SDOS5.0,SDOS5.1,SDOS5.2,SDOS5.3,SDOS5.4
0,Train-01,103.86,CCCCCCCCCCCCc1ccsc1-c1ccc(-c2cccs2)cc1,410.692,2,0,9.6070,0.301361,5.187321,0.00,...,1.717761,1.970186,1.760071,1.224983,0.664733,0.282353,0.096763,0.034589,0.030793,0.057340
1,Train-02,101.13,CCCCCCCCCCCCc1ccsc1-c1cccs1,334.594,2,0,7.9400,0.367472,4.141425,0.00,...,0.012396,0.046031,0.133124,0.299840,0.525958,0.718549,0.764711,0.634854,0.414866,0.225909
2,Train-03,78.30,CN1CCN(S(=O)(=O)c2ccc(-c3ccc(-c4cccs4)cc3)cc2)CC1,398.553,4,0,4.0182,0.799589,5.368024,40.62,...,2.421162,2.703267,2.352276,1.595867,0.845839,0.354620,0.127878,0.060600,0.064782,0.098908
3,Train-04,71.88,O=C1c2ccccc2C(=O)c2cc(-c3ccc(-c4cccs4)s3)ccc21,372.470,4,0,5.9190,0.793825,4.948903,34.14,...,0.886320,0.579059,0.345148,0.246564,0.276259,0.381997,0.495304,0.566935,0.594203,0.614075
4,Train-05,68.37,CC(C)(C)OC(=O)n1ccc2ccc(-c3ccc(-c4ccc(-c5cccs5...,457.620,5,0,8.5485,0.671148,5.994751,31.23,...,0.487723,0.245764,0.249019,0.363222,0.474953,0.505358,0.440671,0.330129,0.234649,0.183111


In [5]:
print(train_df.shape)
train_df.isnull().sum().sort_values(ascending=False).head(20)

(42, 146)


Batch_ID    0
TDOS4.1     0
TDOS2.5     0
TDOS2.6     0
TDOS2.7     0
TDOS2.8     0
TDOS2.9     0
TDOS3.0     0
TDOS3.1     0
TDOS3.2     0
TDOS3.3     0
TDOS3.4     0
TDOS3.5     0
TDOS3.6     0
TDOS3.7     0
TDOS3.8     0
TDOS3.9     0
TDOS2.4     0
TDOS2.3     0
TDOS2.2     0
dtype: int64

In [7]:
display(train_df["Smiles"].info())

<class 'pandas.core.series.Series'>
RangeIndex: 42 entries, 0 to 41
Series name: Smiles
Non-Null Count  Dtype 
--------------  ----- 
42 non-null     object
dtypes: object(1)
memory usage: 468.0+ bytes


None

In [9]:
X = train_df.drop(columns=['Batch_ID','T80','Smiles'],axis=1)
y = train_df["T80"]

In [11]:
models = {
    "RandomForest":RandomForestRegressor(max_depth=5,n_estimators=10_000,verbose=0),
    "GradienBoosting":GradientBoostingRegressor(max_depth=5,n_estimators=10_000,verbose=0),
    "Support Vector Regression":SVR(kernel="rbf",C=100,verbose=0,gamma="scale",epsilon=0.01),
    "SGDR":SGDRegressor(loss='huber',random_state=700,early_stopping=True,epsilon=0.001)}

In [43]:
msle_scorer = make_scorer(mean_squared_log_error, greater_is_better=False)
skf = KFold(n_splits=10, shuffle=True, random_state=700)
y_log = np.log1p(y)

In [83]:
for name, model in models.items():
    # cross_val_score will return negative MSLE scores because we set 
    # greater_is_better=False above
    scores = cross_val_score(
        model,
        X,
        y,
        cv=skf.split(X, y_log),  # custom split with StratifiedKFold
        scoring=msle_scorer
    )
    
    # Convert negative MSLE to positive MSLE
    msle_values = -scores
    print(f"Model {name}:")
    print(f"    -> Fold scores (MSLE): {msle_values}")
    print(f"    -> Average MSLE       : {msle_values.mean():.6f}\n")

Model RandomForest:
    -> Fold scores (MSLE): [1.38912192 0.24055317 0.58847899 2.14226188 1.38519822]
    -> Average MSLE       : 1.149123

Model GradienBoosting:
    -> Fold scores (MSLE): [0.67622012 0.50066656 0.93261704 2.78479608 2.59057591]
    -> Average MSLE       : 1.496975

Model Support Vector Regression:
    -> Fold scores (MSLE): [1.02020037 0.37806305 0.40927181 2.11020328 0.70704823]
    -> Average MSLE       : 0.924957

Model SGDR:
    -> Fold scores (MSLE): [2.50149489 0.96912467 1.7009263  1.16633208 1.5899594 ]
    -> Average MSLE       : 1.585567



In [97]:
kf = KFold(n_splits=5,shuffle=True,random_state=700)
cat = CatBoostRegressor(
        max_depth=5,  
        random_seed=420, 
        subsample=1,
        loss_function='RMSE',
        eval_metric='MSLE',
        n_estimators=10_0000,  
        learning_rate=0.009, 
        verbose=0,
        early_stopping_rounds=10
    )
scores_cat = cross_val_score(cat,X,y_log,cv=kf,scoring=msle_scorer)
msle_values_cat = -scores_cat
print(f"fold msle for catboost : {msle_values_cat}")
print(f"average msle for catboost : {msle_values_cat.mean():.4f}")

fold msle for catboost : [0.09882327 0.02548201 0.04088658 0.11113048 0.09390963]
average msle for catboost : 0.0740


In [107]:
neigh = KNeighborsRegressor(n_neighbors=14,leaf_size=100,p=3,n_jobs=15,weights='uniform')
scores_neigh = cross_val_score(neigh,X,y_log,cv=skf,scoring=msle_scorer)
msle_values_neigh = -scores_neigh
print(f"fold msle for KNNeighbors : {msle_values_neigh}")
print(f"average msle for KNNeighbors : {msle_values_neigh.mean():.4f}")
neigh.fit(X,y_log)

fold msle for KNNeighbors : [0.09976955 0.05556598 0.02865283 0.05458327 0.04816552 0.03051737
 0.13181339 0.15529679 0.04052376 0.07880778]
average msle for KNNeighbors : 0.0724


KNeighborsRegressor(leaf_size=100, n_jobs=15, n_neighbors=14, p=3)

In [89]:
cat.fit(X,y)

In [87]:
test_df = pd.read_csv("test_molecular.csv")

X_test = test_df.drop(columns=['Batch_ID','T80','Smiles'])
# --- 4) Predict with your trained CatBoostRegressor instance (cat_model) ---
y_pred = neigh.predict(X_test)

In [89]:
mole_df = pd.DataFrame({"Batch_ID":test_df["Batch_ID"],"T80":y_pred})
display(mole_df)
mole_df.to_csv("submission_mole.csv",index=False)

,Batch_ID,T80
0,Test-01,3.259932
1,Test-02,2.999859
2,Test-03,3.259932
3,Test-04,2.999859
4,Test-05,2.827606
5,Test-06,3.259932
6,Test-07,3.259932
7,Test-08,3.259932
8,Test-09,2.693299


In [37]:
test_df

,Batch_ID,T80,Smiles,Mass,HAcceptors,HDonors,LogP,Asphericity,Rg,TPSA,...,SDOS4.5,SDOS4.6,SDOS4.7,SDOS4.8,SDOS4.9,SDOS5.0,SDOS5.1,SDOS5.2,SDOS5.3,SDOS5.4
0,Test-01,NaN,COC(=O)c3ccc(c2ccc(c1ccco1)cc2)s3,284.336,4,0,4.4617,0.796737,4.323524,39.44,...,0.709932,0.266678,0.078569,0.020244,0.011813,0.025880,0.061288,0.118430,0.181805,0.220200
1,Test-02,NaN,CCCCCCc2c(CCCCCC)c(c1cccs1)sc2c3ccc(C(=O)OC)cc3,468.728,4,0,9.1758,0.268870,5.019005,26.30,...,1.913133,1.823127,1.356344,0.797689,0.399819,0.236639,0.250003,0.343148,0.426963,0.445557
2,Test-03,NaN,COC(=O)c3ccc(c2ccc(c1cccc(OC)c1)cc2)s3,324.401,4,0,4.8773,0.846029,4.976660,35.53,...,2.302484,1.460937,0.729424,0.299914,0.124335,0.079533,0.080611,0.097277,0.129095,0.178570
3,Test-04,NaN,CCCCCCc3csc(c2ccc(c1ccc(C(=O)OC)s1)cc2)c3CCCCCC,468.728,4,0,9.1758,0.654627,5.609674,26.30,...,1.861847,1.032900,0.447376,0.154937,0.054493,0.048734,0.094820,0.181653,0.293166,0.390163
4,Test-05,NaN,CCCCCCc1ccsc1c3cc(CCCCCC)c(c2sccc2CCCCCC)s3,500.883,3,0,11.5735,0.406478,5.526453,0.00,...,0.432056,0.166929,0.051487,0.016869,0.017613,0.040848,0.094981,0.197741,0.356762,0.543620
5,Test-06,NaN,c4ccc3c(c1ccncc1)ccc(c2ccncc2)c3c4,282.346,2,0,4.9638,0.533503,3.743344,25.78,...,0.777461,1.066911,1.140554,0.950557,0.619796,0.322005,0.148311,0.097106,0.141809,0.270638
6,Test-07,NaN,c4cc(c2ccc(c1ccncc1)c3nsnc23)ccn4,292.367,5,0,3.1617,0.477945,3.506304,51.56,...,0.000012,0.000092,0.000558,0.002758,0.011183,0.037205,0.101123,0.223117,0.397414,0.570078
7,Test-08,NaN,c3cc(c2ccc(c1ccncc1)s2)ccn3,238.315,3,0,3.8721,0.749892,3.629240,25.78,...,1.384438,0.755842,0.325354,0.119390,0.056951,0.062667,0.098769,0.146215,0.184803,0.196689
8,Test-09,NaN,CCCCCCc2c(CCCCCC)c(c1cccc(OC)c1)sc2c3ccc(C(=O)...,492.725,4,0,9.1229,0.269112,5.146173,35.53,...,1.635836,1.974697,1.866746,1.399961,0.869828,0.510805,0.368140,0.373142,0.442061,0.503521


In [71]:
train_df

,Batch_ID,T80,Smiles,Mass,HAcceptors,HDonors,LogP,Asphericity,Rg,TPSA,...,SDOS4.5,SDOS4.6,SDOS4.7,SDOS4.8,SDOS4.9,SDOS5.0,SDOS5.1,SDOS5.2,SDOS5.3,SDOS5.4
0,Train-01,103.86,CCCCCCCCCCCCc1ccsc1-c1ccc(-c2cccs2)cc1,410.692,2,0,9.60700,0.301361,5.187321,0.00,...,1.717761,1.970186,1.760071,1.224983,0.664733,0.282353,0.096763,0.034589,0.030793,0.057340
1,Train-02,101.13,CCCCCCCCCCCCc1ccsc1-c1cccs1,334.594,2,0,7.94000,0.367472,4.141425,0.00,...,0.012396,0.046031,0.133124,0.299840,0.525958,0.718549,0.764711,0.634854,0.414866,0.225909
2,Train-03,78.30,CN1CCN(S(=O)(=O)c2ccc(-c3ccc(-c4cccs4)cc3)cc2)CC1,398.553,4,0,4.01820,0.799589,5.368024,40.62,...,2.421162,2.703267,2.352276,1.595867,0.845839,0.354620,0.127878,0.060600,0.064782,0.098908
3,Train-04,71.88,O=C1c2ccccc2C(=O)c2cc(-c3ccc(-c4cccs4)s3)ccc21,372.470,4,0,5.91900,0.793825,4.948903,34.14,...,0.886320,0.579059,0.345148,0.246564,0.276259,0.381997,0.495304,0.566935,0.594203,0.614075
4,Train-05,68.37,CC(C)(C)OC(=O)n1ccc2ccc(-c3ccc(-c4ccc(-c5cccs5...,457.620,5,0,8.54850,0.671148,5.994751,31.23,...,0.487723,0.245764,0.249019,0.363222,0.474953,0.505358,0.440671,0.330129,0.234649,0.183111
5,Train-06,51.70,c1csc(-c2ccc(-c3ccc(-c4ccc5c(c4)C4(c6ccccc6Oc6...,572.754,3,0,11.27940,0.579470,5.999279,9.23,...,0.341195,0.334334,0.404439,0.437977,0.406705,0.343641,0.292732,0.277376,0.303196,0.365520
6,Train-07,48.00,CC(C)(C)OC(=O)CN1C(=O)CCc2cc(-c3ccc(-c4cccs4)s...,425.575,5,0,5.76450,0.692029,5.358738,46.61,...,0.573623,0.218337,0.074412,0.037166,0.039435,0.053866,0.070479,0.083547,0.092867,0.108030
7,Train-08,44.96,Cc1cc(-c2ccc(-c3ccc(-c4cccs4)s3)cc2)c2nnnn2c1,374.494,6,0,5.55672,0.761473,5.110443,43.08,...,0.159368,0.114065,0.150824,0.197677,0.214038,0.194934,0.164543,0.147707,0.149024,0.161365
8,Train-09,38.74,CCCCCCCCCCCCc1ccsc1-c1ccc(-c2ccc(-c3ccc(-c4ccc...,580.975,5,0,13.12550,0.265031,6.364666,0.00,...,0.308259,0.392212,0.399284,0.329140,0.241205,0.195217,0.202376,0.233241,0.264707,0.314958
9,Train-10,32.17,Cc1ccc(-c2ccc(-c3ccc(-c4cccs4)s3)cc2)cc1Cc1ccc...,522.735,3,0,10.57742,0.718782,6.970584,0.00,...,0.501778,0.465954,0.789773,1.303114,1.732565,1.817389,1.514204,1.028088,0.614496,0.391702


In [67]:
print(X.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 42 entries, 0 to 41
Columns: 144 entries, Mass to Smiles_encoded
dtypes: float64(137), int32(1), int64(6)
memory usage: 47.2 KB
None
